In [102]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Читаем файл

In [103]:
contacts_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\Contacts (Done).xlsx"
contacts = pd.read_excel(contacts_path,dtype={"Id": "string"})

### Приводим названия колонок к удобному формату

In [104]:
contacts.columns = (contacts.columns.str.strip().str.lower().str.replace(" ", "_", regex=False))
contacts.shape

(18548, 4)

In [105]:
contacts.head()

,id,contact_owner_name,created_time,modified_time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14


In [106]:
contacts.dtypes

id                    string[python]
contact_owner_name            object
created_time                  object
modified_time                 object
dtype: object

## Общая статистика по заполненности, типам и уникальности

In [107]:
contacts_info = pd.DataFrame({
    "column": contacts.columns,
    "non_null": contacts.notna().sum().values,
    "missing": contacts.isna().sum().values,
    "missing_pct": (contacts.isna().mean().values * 100).round(2),
    "dtype": contacts.dtypes.astype(str).values,
    "unique_values": contacts.nunique(dropna=True).values})
contacts_info

,column,non_null,missing,missing_pct,dtype,unique_values
0,id,18548,0,0.0,string,18548
1,contact_owner_name,18548,0,0.0,object,28
2,created_time,18548,0,0.0,object,17921
3,modified_time,18548,0,0.0,object,16580


## Работаем с типами данных в created_time и modified_time


In [108]:
date_columns = ["created_time", "modified_time"]

for col in date_columns:
    contacts[col] = pd.to_datetime(contacts[col], dayfirst=True, errors="coerce")

contacts[date_columns].dtypes

created_time     datetime64[ns]
modified_time    datetime64[ns]
dtype: object

In [109]:
# Проверяем, все ли даты корректно преобразовались
for col in date_columns:
    print(f"{col}:")
    print(f"  пустых дат после преобразования: {contacts[col].isna().sum()}")
    print(f"  минимальная дата: {contacts[col].min()}")
    print(f"  максимальная дата: {contacts[col].max()}")

created_time:
  пустых дат после преобразования: 0
  минимальная дата: 2023-06-27 11:28:00
  максимальная дата: 2024-06-21 15:30:00
modified_time:
  пустых дат после преобразования: 0
  минимальная дата: 2023-07-06 10:54:00
  максимальная дата: 2024-06-21 15:32:00


In [110]:
# Проверяем, нет ли случаев, где modified_time раньше created_time
wrong_dates = contacts[contacts["modified_time"] < contacts["created_time"]]
wrong_dates.shape

(0, 4)

## Проверка на дубликаты

In [111]:
# Полные дубликаты по всем столбцам
full_duplicates_count = contacts.duplicated().sum()
full_duplicates_count

np.int64(0)

In [112]:
# Дубликаты по id
id_duplicates_count = contacts["id"].duplicated().sum()
id_duplicates_count

np.int64(0)

In [113]:
# Теперь бизнес дупликаты: контакт может быть дублем, если совпадают все поля, кроме id.
duplicate_subset = ["contact_owner_name", "created_time", "modified_time"]
business_duplicates = contacts[contacts.duplicated(subset=duplicate_subset, keep=False)].sort_values(duplicate_subset)
business_duplicates.shape[0] # Сколько строк участвуют в таких дублях

71

In [114]:
# Сколько строк можно удалить, если оставить одну запись в каждой группе
contacts.duplicated(subset=duplicate_subset).sum()

np.int64(38)

### Таким образом:
* Полных дублей нет.
* Дублей по id нет.
* Есть 38 бизнес-дублей: совпадают contact_owner_name, created_time, modified_time, но отличаются id.
  
#### Так как в Contacts нет телефона/email/имени клиента, это вероятные дубли, но перед удалением нужно сохранить mapping старых id к master_id, чтобы потом не потерять связи в Calls и Deals

### Создаем mapping для бизнес-дублей
Маппинг — это таблица соответствий.
В нашем случае она отвечает на вопрос:
Какой старый contact_id нужно заменить на какой главный contact_id. В Contacts есть дубли: один и тот же контакт записан несколько раз, но с разными id.
А вдруг мы этот контакт удалили. Тогда звонок “оторвется” от контакта. И потом в Calls и Deals сможем заменить старые ID на главный ID.

In [115]:
duplicate_subset = ["contact_owner_name", "created_time", "modified_time"]

duplicate_rows = contacts[contacts.duplicated(subset=duplicate_subset, keep=False)].copy()

mapping_rows = []

for _, group in duplicate_rows.groupby(duplicate_subset, dropna=False):
    ids = sorted(group["id"].dropna().astype(str).tolist())
    master_id = ids[0]

    for old_id in ids[1:]:
        mapping_rows.append({
            "old_contact_id": old_id,
            "master_contact_id": master_id,
            "contact_owner_name": group["contact_owner_name"].iloc[0],
            "created_time": group["created_time"].iloc[0],
            "modified_time": group["modified_time"].iloc[0], })

contacts_mapping = pd.DataFrame(mapping_rows)

# Удаляем бизнес-дубли из Contacts. Оставляем master_id — минимальный id в каждой группе
contacts_clean = (contacts.sort_values("id").drop_duplicates(subset=duplicate_subset, keep="first").reset_index(drop=True))

print("Строк до обработки:", len(contacts))
print("Строк в группах дублей:", len(duplicate_rows))
print("Удаляемых дублей:", len(contacts_mapping))
print("Строк после обработки:", len(contacts_clean))

Строк до обработки: 18548
Строк в группах дублей: 71
Удаляемых дублей: 38
Строк после обработки: 18510


In [116]:
# Проверяем один контакт, у которого менеджер указан как False
false_contact = contacts[contacts["contact_owner_name"].astype("string").str.lower() == "false"]
false_contact

,id,contact_owner_name,created_time,modified_time
2197,5805028000008772190,False,2023-09-24 09:01:00,2023-10-13 16:44:00


In [117]:
# Сохраняем id этого контакта
false_contact_id = false_contact["id"].iloc[0]
false_contact_id

'5805028000008772190'

In [118]:
# Проверяем этот контакт в Calls
calls_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\Calls (Done).xlsx"
calls = pd.read_excel(calls_path, dtype={"Id": "string", "CONTACTID": "string"})
calls_for_false_contact = calls[calls["CONTACTID"].astype("string").str.strip() == false_contact_id]
calls_for_false_contact

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
9721,5805028000008771348,24.09.2023 11:43,George King,5805028000008772190,Outbound,1208.0,Attended Dialled,NaN,Completed,0.0,NaN


In [119]:
# Проверяем этот контакт в Deals
deals_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\Deals (Done).xlsx"
deals = pd.read_excel(deals_path, dtype={"Id": "string", "Contact Name": "string"})
deals_for_false_contact = deals[deals["Contact Name"].astype("string").str.strip() == false_contact_id]
deals_for_false_contact

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
19112,5805028000008755203,Charlie Davis,19.10.2023,C - Low,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,02:41:21,v9com,...,NaN,NaN,24.09.2023 09:01,NaN,NaN,NaN,NaN,5805028000008772190,NaN,NaN


In [120]:
# Исправляем менеджера False на Charlie Davis. Решение принято по Deals (так как звонить могут многие, а реальный контакт за тем кто ведет сделку клиента).
contacts.loc[contacts["contact_owner_name"].astype("string").str.lower() == "false", "contact_owner_name"] = "Charlie Davis"

In [121]:
# Проверяем, что False больше нет
contacts["contact_owner_name"].value_counts(dropna=False).tail(30)

contact_owner_name
Charlie Davis      2019
Ulysses Adams      1816
Julia Nelson       1769
Paula Underwood    1487
Quincy Vincent     1416
Nina Scott         1150
Ben Hall           1038
Victor Barnes       967
Cara Iverson        880
Rachel White        782
Jane Smith          754
Bob Brown           685
Ian Miller          684
Diana Evans         678
Yara Edwards        655
Amy Green           621
Eva Kent            365
Kevin Parker        325
Mason Roberts       217
George King         144
Sam Young            37
Alice Johnson        27
Oliver Taylor        19
Zachary Foster        8
Wendy Clark           2
Tina Zhang            2
Derek James           1
Name: count, dtype: int64

In [122]:
 # Добавляем contact_lifetime_days
contacts_clean["contact_lifetime_days"] = (contacts_clean["modified_time"] - contacts_clean["created_time"]).dt.total_seconds() / 86400
contacts_clean[["created_time", "modified_time", "contact_lifetime_days"]].head()

,created_time,modified_time,contact_lifetime_days
0,2023-06-27 11:28:00,2023-12-22 13:34:00,178.087500
1,2023-07-03 11:31:00,2024-05-21 10:23:00,322.952778
2,2023-07-02 22:37:00,2023-12-21 13:17:00,171.611111
3,2023-07-03 05:44:00,2023-12-29 15:20:00,179.400000
4,2023-07-04 10:11:00,2024-04-16 16:14:00,287.252083


In [123]:
# Считаем среднее и медианное время жизни контакта по менеджеру (для поверхностной аналитики)
manager_lifetime = (contacts_clean.groupby("contact_owner_name", dropna=False).agg(
        manager_contacts_count=("id", "count"),
        manager_avg_contact_lifetime_days=("contact_lifetime_days", "mean"),
        manager_median_contact_lifetime_days=("contact_lifetime_days", "median")).reset_index())

manager_lifetime.sort_values("manager_contacts_count", ascending=False).head()

,contact_owner_name,manager_contacts_count,manager_avg_contact_lifetime_days,manager_median_contact_lifetime_days
6,Charlie Davis,2018,21.718168,0.085417
23,Ulysses Adams,1809,37.286757,0.262500
13,Julia Nelson,1769,25.195877,0.086806
18,Paula Underwood,1486,15.014638,0.085417
19,Quincy Vincent,1415,13.006094,0.084722


# Итоговая проверка после очистки

In [125]:
print("Строк исходно:", len(contacts))
print("Строк после очистки:", len(contacts_clean))
print("Удалено бизнес-дублей:", len(contacts) - len(contacts_clean))
print("Исправлено значений False:", 1)

print("\nПропуски:")
display(contacts_clean.isna().sum())

print("\nПолные дубли:", contacts_clean.duplicated().sum())
print("Дубли по id:", contacts_clean["id"].duplicated().sum())
print("Бизнес-дубли:", contacts_clean.duplicated(subset=duplicate_subset).sum())

print("Ошибки дат modified_time < created_time:", (contacts_clean["modified_time"] < contacts_clean["created_time"]).sum())
print("Уникальных менеджеров:", contacts_clean["contact_owner_name"].nunique())

Строк исходно: 18548
Строк после очистки: 18510
Удалено бизнес-дублей: 38
Исправлено значений False: 1

Пропуски:


id                       0
contact_owner_name       0
created_time             0
modified_time            0
contact_lifetime_days    0
dtype: int64


Полные дубли: 0
Дубли по id: 0
Бизнес-дубли: 0
Ошибки дат modified_time < created_time: 0
Уникальных менеджеров: 28


# Сохраняем результат

In [126]:
output_dir = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы"
os.makedirs(output_dir, exist_ok=True)
contacts_clean.to_csv(os.path.join(output_dir, "contacts_clean.csv"), index=False, encoding="utf-8-sig")
contacts_clean.to_excel(os.path.join(output_dir, "contacts_clean.xlsx"), index=False)
contacts_mapping.to_csv(os.path.join(output_dir, "contacts_duplicates_mapping.csv"), index=False, encoding="utf-8-sig")
manager_lifetime.to_csv(os.path.join(output_dir, "contacts_manager_lifetime_summary.csv"), index=False, encoding="utf-8-sig")

## Вывод по очистке Contacts

* В таблице Contacts было 18 548 строк и 4 столбца. 
* Пропусков, полных дублей и дублей по `id` не обнаружено. 
* ID были сохранены в строковом формате, чтобы избежать потери точности при дальнейших объединениях таблиц.

* Были найдены бизнес-дубли: контакты с разными `id`, но одинаковыми `contact_owner_name`, `created_time` и `modified_time`. Таких строк к удалению оказалось 38. Для них был создан mapping `old_contact_id -> master_contact_id`, чтобы позже корректно перепривязать звонки и сделки к основному контакту в основном файле Deals.

* Также было найдено одно некорректное значение менеджера `False`. После проверки в Calls и Deals оно было заменено на `Charlie Davis`, так как в Deals именно этот менеджер указан владельцем сделки по данному контакту.

* Дополнительно были создан признак `contact_lifetime_days`, который может быть использованы для дальнейшего анализа работы менеджеров и связи длительности жизни контакта с результатом продаж.